# Prompt 08: Environment test (Version A, T21)

Vaibhav Sharma, MSE2026010

Run every cell in order on a **T4 GPU** runtime (Runtime > Change runtime type > T4 GPU). After the last cell, copy the full output of cells 2, 4, 6 and 7 and paste it back to the assistant. Do not edit outputs.

In [ ]:
# Cell 1: upload the Prompt 08 zip (VersionA_Prompt08_code.zip) and unzip it
from google.colab import files
up = files.upload()
import zipfile, os
for n in up:
    if n.endswith('.zip'):
        zipfile.ZipFile(n).extractall('p08')
os.chdir('p08')
print(os.getcwd(), os.listdir('.'))

In [ ]:
# Cell 2: GPU and Python diagnostics
!nvidia-smi
!python --version
import torch
print('torch', torch.__version__, '| CUDA build', torch.version.cuda, '| GPU available', torch.cuda.is_available())

In [ ]:
# Cell 3: install proposed dependencies (lower bounds, untested)
!pip install -q -r requirements.in

In [ ]:
# Cell 4: compatibility check (should report no broken requirements)
!pip check

In [ ]:
# Cell 5: load secrets from Colab Secrets (key icon on the left). Add OPENALEX_API_KEY, CONTACT_EMAIL, and HF_TOKEN if needed.
import os
from google.colab import userdata
for k in ('OPENALEX_API_KEY', 'CONTACT_EMAIL', 'HF_TOKEN'):
    try:
        os.environ[k] = userdata.get(k) or ''
    except Exception:
        os.environ[k] = ''
    print(k, 'set' if os.environ[k] else 'NOT set')

In [ ]:
# Cell 6: environment test (writes results/env_report.json)
!python scripts/env_test.py

In [ ]:
# Cell 7: inspect Naser's released dataset (needed for Prompts 06 and 09)
import os
from huggingface_hub import list_repo_files
rid = 'mznaser/llm-citation-hallucination-audit'
tok = os.environ.get('HF_TOKEN') or None
try:
    fs = list_repo_files(rid, repo_type='dataset', token=tok)
    print('FILES:', fs)
    import pandas as pd
    from huggingface_hub import hf_hub_download
    for f in fs:
        if f.endswith(('.csv', '.parquet', '.jsonl', '.json')):
            p = hf_hub_download(rid, f, repo_type='dataset', token=tok)
            if f.endswith('.csv'):
                d = pd.read_csv(p)
            elif f.endswith('.parquet'):
                d = pd.read_parquet(p)
            else:
                d = pd.read_json(p, lines=f.endswith('.jsonl'))
            print('\n==', f, d.shape)
            print('COLUMNS:', list(d.columns))
            print(d.head(3).to_string()[:1500])
except Exception as e:
    print('COULD NOT OPEN DATASET:', type(e).__name__, str(e)[:300])
    print('Next: try the GitHub repo github.com/mznaser-clemson/llm-citation-hallucination-audit in a browser, and report what you see.')

In [ ]:
# Cell 8: freeze the versions that actually worked, and show the report
!pip freeze > requirements.txt
!wc -l requirements.txt
!cat results/env_report.json

In [ ]:
# Cell 9 (optional): download the outputs to commit to your GitHub repo
from google.colab import files
files.download('results/env_report.json')
files.download('requirements.txt')